In [ ]:
import pyodbc
import pandas as pd
import matplotlib.pyplot as plt
import plotly.express as px

In [ ]:
server = 'sharktowels.duckdns.org'
database = "Spending"
user = "SA"

with open("password.txt", "r") as file:
    password = file.read().strip()

connection_string = (
    f"DRIVER={{ODBC Driver 18 for SQL Server}};"
    f"SERVER={server};"
    f"DATABASE={database};"
    f"UID={user};"
    f"PWD={password};"
    "TrustServerCertificate=yes;"
)

conn = pyodbc.connect(connection_string)


C:\Users\jduen\AppData\Local\Temp\ipykernel_4852\1637103813.py:19: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df = pd.read_sql("SELECT * FROM Purchases", conn)


In [44]:
user = 12
area_df = pd.read_sql_query(f'''
                            SELECT Purchases.TimeDate, Purchases.Category, Purchases.Subcategory, Payments.Amount 
                            FROM Purchases
                            JOIN Payments ON Purchases.PaymentID = Payments.PaymentID
                            WHERE UserID = {user}
                            ORDER BY Purchases.TimeDate
                            ''', conn)

area_df["runningAmount"] = area_df.sort_values("TimeDate") \
                         .groupby("Category")["Amount"] \
                         .cumsum()

pivoted_area_df = area_df.pivot_table(
    index="TimeDate",
    columns="Category",
    values="runningAmount",
    aggfunc="last"
)

pivoted_area_df = pivoted_area_df.ffill()

pivoted_area_df.head()

C:\Users\jduen\AppData\Local\Temp\ipykernel_4852\2537624565.py:2: UserWarning:

pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.



Category,entertainment,food,future,health,housing,insurance,miscellaneous,supplies,transportation,utilities
TimeDate,,,,,,,,,,
2023-12-22 02:55:00,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,235.12
2024-01-19 02:55:00,NaN,NaN,NaN,165.05,NaN,NaN,NaN,NaN,NaN,235.12
2024-03-16 02:55:00,NaN,NaN,NaN,165.05,NaN,NaN,17.65,NaN,NaN,235.12
2024-03-29 02:55:00,NaN,NaN,NaN,165.05,NaN,NaN,273.16,NaN,NaN,235.12
2024-04-13 02:55:00,NaN,NaN,103.07,165.05,NaN,NaN,273.16,NaN,NaN,235.12


In [ ]:
fig = px.area(pivoted_area_df, x=pivoted_area_df.index, y=pivoted_area_df.columns)
fig.show()